In [22]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [23]:
# Unscaled datasets
train_unscaled = pd.read_csv("train_unscaled_processed.csv")
test_unscaled = pd.read_csv("test_unscaled_processed.csv")

# Scaled datasets
train_scaled = pd.read_csv("train_scaled_processed.csv")
test_scaled = pd.read_csv("test_scaled_processed.csv")

In [24]:
X_train_unscaled = train_unscaled.drop(["Fatigue", "FatigueIndex"], axis=1)
X_test_unscaled = test_unscaled.drop(["Fatigue", "FatigueIndex"], axis=1)

X_train_scaled = train_scaled.drop(["Fatigue", "FatigueIndex"], axis=1)
X_test_scaled = test_scaled.drop(["Fatigue", "FatigueIndex"], axis=1)

In [25]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train_unscaled, y_train)

rf_pred = rf_model.predict(X_test_unscaled)

In [31]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# Define model
lr = LogisticRegression(max_iter=2000,C= 10, multi_class= 'multinomial', penalty= 'l2', solver= 'lbfgs')
lr.fit(X_train_scaled, y_train)

lr_pred = lr.predict(X_test_scaled)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [27]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    eval_metric='mlogloss'
)

xgb_model.fit(X_train_unscaled, y_train)

xgb_pred = xgb_model.predict(X_test_unscaled)

In [32]:
from sklearn.metrics import accuracy_score, classification_report

print("RF Accuracy:", accuracy_score(y_test, rf_pred))
print("LR Accuracy:", accuracy_score(y_test, lr_pred))
print("XGB Accuracy:", accuracy_score(y_test, xgb_pred))

RF Accuracy: 0.9335714285714286
LR Accuracy: 0.735
XGB Accuracy: 0.9642857142857143


In [39]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# -------------------------------
# LOAD TRAIN DATA
# -------------------------------
df = pd.read_csv("train_data.csv")

# -------------------------------
# CLEAN TARGET (IMPORTANT)
# -------------------------------
df['Fatigue'] = df['Fatigue'].astype(str).str.strip().str.lower()
df['Fatigue'] = df['Fatigue'].map({
    'low': 0,
    'medium': 1,
    'high': 2
})

# -------------------------------
# SAVE GLOBAL STATS (VERY IMPORTANT)
# -------------------------------
fe_config = {
    "activity_max": df['ActivityLevel'].max(),
    "sitting_max": df['SittingTime'].max(),
    "inactivity_max": df['InactivityPeriods'].max(),
    "meals_max": df['MealsPerDay'].max(),
    "diet_max": df['DietQuality'].max(),
    "stress_max": df['StressLevel'].max(),
    "dopamine_max": df['dopamine_score'].max(),
    "latenight_max": (df['LateNightUsage'] * df['ScreenTime']).max()
}

joblib.dump(fe_config, "fe_config.pkl")

# -------------------------------
# FEATURE ENGINEERING FUNCTION
# -------------------------------
def feature_engineering(df):

    df = df.copy()

    # Load config
    config = joblib.load("fe_config.pkl")

    # ---- Gender ----
    df['Gender'] = df['Gender'].astype(str).str.strip().str.lower()
    df['Gender'] = df['Gender'].map({'male': 0, 'female': 1}).fillna(0)

    # ---- BMI Category ----
    def bmi_category(bmi):
        if bmi < 18.5:
            return 0
        elif bmi < 25:
            return 1
        elif bmi < 30:
            return 2
        else:
            return 3

    df['bmi_category'] = df['BMI'].apply(bmi_category)

    # ---- Sleep ----
    df['SleepScore'] = 1 - abs(df['SleepHours'] - 7.5) / 7.5

    # ---- Digital ----
    df['DigitalLoad'] = df['ScreenTime']
    df['LateNightImpact'] = df['LateNightUsage'] * df['ScreenTime']

    # ---- Activity ----
    df['ActivityScore'] = df['ActivityLevel'] / config["activity_max"]

    df['SedentaryIndex'] = (
        df['SittingTime'] + df['InactivityPeriods']
    ) / (config["sitting_max"] + config["inactivity_max"])

    # ---- Diet ----
    df['CalorieBalance'] = df['CalorieIntake'] / 2000
    df['MealScore'] = df['MealsPerDay'] / config["meals_max"]
    df['DietScore'] = df['DietQuality'] / config["diet_max"]

    # ---- Stress ----
    df['StressScore'] = df['StressLevel'] / config["stress_max"]

    # ---- Dopamine ----
    df['DopamineScore'] = df['dopamine_score'] / config["dopamine_max"]

    # ---- Handle Missing ----
    df.fillna(df.mean(numeric_only=True), inplace=True)

    return df


# -------------------------------
# APPLY FEATURE ENGINEERING
# -------------------------------
df = feature_engineering(df)

# Remove leakage
if "FatigueIndex" in df.columns:
    df = df.drop(columns=["FatigueIndex"])

# -------------------------------
# SPLIT
# -------------------------------
X = df.drop("Fatigue", axis=1)
y = df["Fatigue"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# -------------------------------
# SCALER
# -------------------------------
scaler = StandardScaler()
scaler.fit(X_train)

# -------------------------------
# SAVE EVERYTHING
# -------------------------------
joblib.dump(feature_engineering, "feature_engineering.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(X_train.columns.tolist(), "features.pkl")

print("✅ Feature Engineering + Scaler saved successfully!")

✅ Feature Engineering + Scaler saved successfully!


In [40]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# -------------------------------
# LOAD TRAIN DATA
# -------------------------------
df = pd.read_csv("train_data.csv")

# -------------------------------
# CLEAN TARGET (IMPORTANT)
# -------------------------------
df['Fatigue'] = df['Fatigue'].astype(str).str.strip().str.lower()
df['Fatigue'] = df['Fatigue'].map({
    'low': 0,
    'medium': 1,
    'high': 2
})

# -------------------------------
# SAVE GLOBAL STATS (VERY IMPORTANT)
# -------------------------------
fe_config = {
    "activity_max": df['ActivityLevel'].max(),
    "sitting_max": df['SittingTime'].max(),
    "inactivity_max": df['InactivityPeriods'].max(),
    "meals_max": df['MealsPerDay'].max(),
    "diet_max": df['DietQuality'].max(),
    "stress_max": df['StressLevel'].max(),
    "dopamine_max": df['dopamine_score'].max(),
    "latenight_max": (df['LateNightUsage'] * df['ScreenTime']).max()
}

joblib.dump(fe_config, "fe_config.pkl")

# -------------------------------
# FEATURE ENGINEERING FUNCTION
# -------------------------------
def feature_engineering(df):

    df = df.copy()

    # Load config
    config = joblib.load("fe_config.pkl")

    # ---- Gender ----
    df['Gender'] = df['Gender'].astype(str).str.strip().str.lower()
    df['Gender'] = df['Gender'].map({'male': 0, 'female': 1}).fillna(0)

    # ---- BMI Category ----
    def bmi_category(bmi):
        if bmi < 18.5:
            return 0
        elif bmi < 25:
            return 1
        elif bmi < 30:
            return 2
        else:
            return 3

    df['bmi_category'] = df['BMI'].apply(bmi_category)

    # ---- Sleep ----
    df['SleepScore'] = 1 - abs(df['SleepHours'] - 7.5) / 7.5

    # ---- Digital ----
    df['DigitalLoad'] = df['ScreenTime']
    df['LateNightImpact'] = df['LateNightUsage'] * df['ScreenTime']

    # ---- Activity ----
    df['ActivityScore'] = df['ActivityLevel'] / config["activity_max"]

    df['SedentaryIndex'] = (
        df['SittingTime'] + df['InactivityPeriods']
    ) / (config["sitting_max"] + config["inactivity_max"])

    # ---- Diet ----
    df['CalorieBalance'] = df['CalorieIntake'] / 2000
    df['MealScore'] = df['MealsPerDay'] / config["meals_max"]
    df['DietScore'] = df['DietQuality'] / config["diet_max"]

    # ---- Stress ----
    df['StressScore'] = df['StressLevel'] / config["stress_max"]

    # ---- Dopamine ----
    df['DopamineScore'] = df['dopamine_score'] / config["dopamine_max"]

    # ---- Handle Missing ----
    df.fillna(df.mean(numeric_only=True), inplace=True)

    return df


# -------------------------------
# APPLY FEATURE ENGINEERING
# -------------------------------
df = feature_engineering(df)

# Remove leakage
if "FatigueIndex" in df.columns:
    df = df.drop(columns=["FatigueIndex"])

# -------------------------------
# SPLIT
# -------------------------------
X = df.drop("Fatigue", axis=1)
y = df["Fatigue"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# -------------------------------
# SCALER
# -------------------------------
scaler = StandardScaler()
scaler.fit(X_train)

# -------------------------------
# SAVE EVERYTHING
# -------------------------------
joblib.dump(feature_engineering, "feature_engineering.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(X_train.columns.tolist(), "features.pkl")

print("✅ Feature Engineering + Scaler saved successfully!")

✅ Feature Engineering + Scaler saved successfully!


In [41]:
joblib.dump(rf_model, "rf_model.pkl")
joblib.dump(lr_model, "lr_model.pkl")
joblib.dump(xgb_model, "xgb_model.pkl")

['xgb_model.pkl']